In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# 输入状态
class InputState(TypedDict):
    username: str

# 输出状态
class OutputState(TypedDict):
    graph_output: str

# 全局状态
class OverAllState(TypedDict):
    username: str
    graph_output: str
    nickname: str

# 私有状态
class PrivateState(TypedDict):
    greeting: str

# 节点1 对接start => InputState 修改的状态内容在全局状态中 => OverAllState
def node_1(state: InputState) -> OverAllState:
    # 向全局状态添加字段nickname
    return {
        "nickname": "Dear " + state["username"],
    }

# 节点2 对接 node_1 => OverAllState 使用的参数在全局状态中 修改的参数在私有状态中 => PrivateState
def node_2(state: OverAllState) -> PrivateState:
    # 向私有状态添加字段greeting
    return {
        "greeting": "你好，" + state["nickname"],
    }

# 节点3 对接 node_2 => PrivateState 使用的参数在私有状态中 修改的参数在输出状态中 => OutputState
def node_3(state: PrivateState) -> OutputState:
    # 向输出状态添加字段graph_output
    return {
        "graph_output": state["greeting"] + "，很高兴认识你!"
    }

# 构建状态图
# 定义图的时候 加载全局状态、输入状态、输出状态
builder = StateGraph(state_schema=OverAllState,input_schema=InputState,output_schema=OutputState)

# 添加节点 加载私有状态
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 添加边
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
# 填入的输入状态
result = graph.invoke({"username": "老吴"})
# 打印输出状态
print(result)


{'graph_output': '你好，Dear 老吴，很高兴认识你!'}
